# LeetCode 802 — Find Eventual Safe States bằng Python

Link đề: https://leetcode.com/problems/find-eventual-safe-states/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Graph, Topological Sort (Kosaraju, Tarjan)

Skeleton LeetCode:

```python
class Solution:
    def eventualSafeNodes(self, graph: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so giữa các cách giải khác nhau, so với cách vét cạn "bao đóng đạt tới" và với các tính chất bắt buộc của tập nút an toàn), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ. Các ví dụ ngoài hai ví dụ chính của đề là ví dụ **tự dựng** (đối chiếu lại đề gốc nếu cần).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đồ thị **có hướng** `n` nút; **nút cuối (terminal)** là nút không có cạnh ra; **nút an toàn (safe)** là nút mà **mọi đường đi** xuất phát từ nó đều dẫn tới nút cuối (hoặc nút an toàn khác). Trả về mọi nút an toàn, **sắp tăng dần**.
2. Nắm các công cụ Python: danh sách kề (`graph[i]` là list các nút kề), `enumerate`, đồ thị **đảo chiều** (`rg[j].append(i)`), `collections.deque` (BFS / Kahn), mảng bậc ra `outdeg`, đệ quy DFS và **ba màu** (chưa thăm / đang thăm / xong), giới hạn đệ quy của Python.
3. Đi từ **bao đóng đạt tới (vét cạn `O(n·(n+m))`)** → **DFS tô ba màu `O(n+m)`** → **Kahn trên đồ thị đảo chiều (sắp xếp tô-pô, `O(n+m)`)** → **Tarjan tìm thành phần liên thông mạnh rồi lan truyền ngược**.
4. Hiểu mấu chốt: nút **không an toàn** ⇔ nó **đạt tới một chu trình** (kể cả khuyên/self-loop). Vậy nút an toàn là những nút **không đạt tới chu trình nào**; ta "gọt" dần: nút cuối an toàn, và một nút mà **mọi nút kề của nó an toàn** thì nó an toàn. Đảo chiều các cạnh và gọt theo bậc ra bằng `0` chính là sắp xếp tô-pô.
5. Tránh các lỗi kinh điển: chỉ loại các nút **nằm trên** chu trình (quên các nút **dẫn tới** chu trình), dùng một tập `visited` không phân biệt "đang thăm" với "đã xong", chạy Kahn trên đồ thị **chưa đảo**, quên **sắp xếp** kết quả, và bỏ qua **khuyên** (cạnh `i → i`).

## 2. Đề bài & ví dụ

Có một đồ thị **có hướng** gồm `n` nút được đánh nhãn từ `0` đến `n - 1`. Đồ thị được cho bởi mảng hai chiều `graph` (đánh chỉ số từ `0`), trong đó `graph[i]` là mảng các nút **kề** của nút `i`, nghĩa là có cạnh từ `i` tới mỗi nút trong `graph[i]`.

Một nút là **nút cuối (terminal node)** nếu **không có cạnh đi ra**. Một nút là **nút an toàn (safe node)** nếu **mọi đường đi có thể** xuất phát từ nó đều dẫn tới một nút cuối (hoặc tới một nút an toàn khác).

Trả về một mảng gồm **tất cả các nút an toàn** của đồ thị. Đáp án phải được **sắp theo thứ tự tăng dần**.

**Ví dụ 1**

- Input: `graph = [[1,2],[2,3],[5],[0],[5],[],[]]`
- Output: `[2,4,5,6]`
- Giải thích: nút `5` và `6` là nút cuối (không có cạnh ra). Mọi đường đi xuất phát từ `2`, `4`, `5`, `6` đều dẫn tới `5` hoặc `6`. Các nút `0`, `1`, `3` tạo thành chu trình `0 → 1 → 3 → 0` nên không an toàn.

**Ví dụ 2**

- Input: `graph = [[1,2,3,4],[1,2],[3,4],[0,4],[]]`
- Output: `[4]`
- Giải thích: chỉ có nút `4` là nút cuối, và mọi đường đi xuất phát từ `4` dẫn tới `4`. Các nút còn lại đều đạt tới chu trình (ví dụ `1` có khuyên `1 → 1`, `0 → 2 → 3 → 0`).

**Ràng buộc:**

- `n == graph.length`, `1 <= n <= 10^4`
- `0 <= graph[i].length <= n`, `0 <= graph[i][j] <= n - 1`
- `graph[i]` được sắp **tăng ngặt** (không có cạnh lặp).
- Đồ thị **có thể có khuyên** (self-loop).
- Số cạnh nằm trong khoảng `[1, 4 * 10^4]`.

Chú ý: "đường đi" có thể **dài vô hạn** (đi vòng mãi trong chu trình); khi đó nó **không** dẫn tới nút cuối. Vì vậy chỉ cần **một** đường đi dẫn vào chu trình là nút đó không an toàn.

In [1]:
from typing import List, Optional
from collections import deque
import itertools
import random
import sys

ex1 = [[1, 2], [2, 3], [5], [0], [5], [], []]
ex2 = [[1, 2, 3, 4], [1, 2], [3, 4], [0, 4], []]
examples = [
    (ex1, [2, 4, 5, 6]),
    (ex2, [4]),
]
extra_examples = [  # tự dựng, đã kiểm chứng; nên đối chiếu lại đề gốc
    ([[]], [0]),                          # một nút cuối duy nhất
    ([[0]], []),                          # khuyên
    ([[1], [1]], []),                     # 0 dẫn vào khuyên ở 1
    ([[1, 2], [3], [3, 4], [], [2]], [1, 3]),   # cặp 2 <-> 4 tạo chu trình
]
for g, expected in examples + extra_examples:
    print(g, "-> kỳ vọng", expected)

[[1, 2], [2, 3], [5], [0], [5], [], []] -> kỳ vọng [2, 4, 5, 6]
[[1, 2, 3, 4], [1, 2], [3, 4], [0, 4], []] -> kỳ vọng [4]
[[]] -> kỳ vọng [0]
[[0]] -> kỳ vọng []
[[1], [1]] -> kỳ vọng []
[[1, 2], [3], [3, 4], [], [2]] -> kỳ vọng [1, 3]


## 3. Kiến thức nền

### 3.1. Đồ thị có hướng bằng danh sách kề

`graph[i]` là list các nút mà `i` **trỏ tới**. `len(graph)` là số nút `n`; `len(graph[i])` là **bậc ra** (số cạnh đi ra) của `i`. Nút có `graph[i] == []` là **nút cuối**. Số cạnh `m` là tổng độ dài các `graph[i]`.

In [2]:
graph = ex1
n = len(graph)
m = sum(len(vs) for vs in graph)
print("n =", n, "m =", m)                                                    # n = 7 m = 7
print("bậc ra:", [len(vs) for vs in graph])                                  # [2, 2, 1, 1, 1, 0, 0]
print("nút cuối:", [i for i, vs in enumerate(graph) if not vs])               # [5, 6]
for i, vs in enumerate(graph):
    print(f"{i} -> {vs}")

n = 7 m = 7
bậc ra: [2, 2, 1, 1, 1, 0, 0]
nút cuối: [5, 6]
0 -> [1, 2]
1 -> [2, 3]
2 -> [5]
3 -> [0]
4 -> [5]
5 -> []
6 -> []


### 3.2. Đường đi, chu trình và khuyên

Một **chu trình** là đường đi quay về nút xuất phát, ví dụ `0 → 1 → 3 → 0`. Một **khuyên** (self-loop) là chu trình độ dài `1`: cạnh `i → i`. Nếu từ một nút ta đi vào được một chu trình thì ta có thể **đi vòng mãi**, và đường đi đó **không** dẫn tới nút cuối. Vì thế, một nút là **không an toàn** nếu và chỉ nếu nó **đạt tới được** một chu trình (có thể chính nó nằm trên chu trình).

In [3]:
def reachable(graph, start):
    """Tập nút đạt tới được từ start bằng ít nhất 1 cạnh (không tính start trừ khi quay về)."""
    seen = set()
    stack = list(graph[start])
    while stack:
        v = stack.pop()
        if v in seen:
            continue
        seen.add(v)
        stack.extend(graph[v])
    return seen


print(reachable(ex1, 0))      # {0, 1, 2, 3, 5}: 0 đạt tới chính nó (nằm trên chu trình)
print(reachable(ex1, 4))      # {5}
print(0 in reachable(ex1, 0), 4 in reachable(ex1, 4))   # True False

{0, 1, 2, 3, 5}
{5}
True False


### 3.3. Đồ thị đảo chiều (reverse graph)

Đồ thị đảo chiều `rg` có cạnh `j → i` **khi và chỉ khi** đồ thị gốc có `i → j`. Trong `rg`, `rg[j]` liệt kê **các nút trỏ tới `j`** ("cha" của `j`). Nó giúp ta "lan ngược" từ nút cuối lên các nút phụ thuộc vào nó.

In [4]:
def reverse_graph(graph):
    rg = [[] for _ in graph]
    for i, vs in enumerate(graph):
        for j in vs:
            rg[j].append(i)
    return rg


rg = reverse_graph(ex1)
print(rg)     # [[3], [0], [0, 1], [1], [], [2, 4], []]

[[3], [0], [0, 1], [1], [], [2, 4], []]


### 3.4. DFS đệ quy với **ba màu**

Khi duyệt DFS để tìm chu trình, mỗi nút có ba trạng thái: **trắng** (`0`, chưa thăm), **xám** (`1`, đang nằm trên đường đi hiện tại hoặc đã biết là xấu) và **đen** (`2`, đã xử lý xong, an toàn). Gặp một nút **xám** nghĩa là **quay lại một nút trên đường đi hiện tại** ⇒ có chu trình. Chỉ dùng một tập `visited` (hai trạng thái) thì **không phân biệt** được "đang trên đường đi" với "đã xong", dẫn đến sai (xem lỗi 2).

In [5]:
def dfs_demo(graph):
    color = [0] * len(graph)

    def dfs(u, depth=0):
        pad = "  " * depth
        color[u] = 1
        print(f"{pad}vào {u} (xám)")
        for v in graph[u]:
            if color[v] == 1:
                print(f"{pad}  gặp {v} đang xám -> có chu trình")
                return False
            if color[v] == 0 and not dfs(v, depth + 1):
                return False
        color[u] = 2
        print(f"{pad}xong {u} (đen)")
        return True

    for u in range(len(graph)):
        if color[u] == 0:
            dfs(u)
    return color


print(dfs_demo([[1], [2], []]))      # chuỗi 0 -> 1 -> 2, tất cả đen

vào 0 (xám)
  vào 1 (xám)
    vào 2 (xám)
    xong 2 (đen)
  xong 1 (đen)
xong 0 (đen)
[2, 2, 2]


### 3.5. Hàng đợi `deque` và bậc

`deque` cho `append` (thêm cuối) và `popleft` (lấy đầu) với `O(1)`. Thuật toán Kahn dùng một hàng đợi các nút **bậc bằng `0`**: lấy một nút ra, **giảm bậc** các nút liên quan, nút nào về `0` thì thêm vào hàng đợi. Ta sẽ dùng **bậc ra** trên đồ thị gốc (tương đương **bậc vào** trên đồ thị đảo).

In [6]:
q = deque([5, 6])
q.append(2)
print(q.popleft(), list(q))      # 5 [6, 2]

outdeg = [len(vs) for vs in ex1]
print(outdeg)                     # [2, 2, 1, 1, 1, 0, 0]

5 [6, 2]
[2, 2, 1, 1, 1, 0, 0]


### 3.6. Giới hạn đệ quy

Python mặc định cho đệ quy sâu khoảng `1000` tầng (`sys.getrecursionlimit()`). Đồ thị có `10^4` nút có thể là **một chuỗi dài** `0 → 1 → ... → 9999`, nên DFS đệ quy có thể báo `RecursionError`. Ta sẽ có phiên bản DFS **không đệ quy** (ngăn xếp tường minh) và cách Kahn (không cần đệ quy) cho code nộp.

In [7]:
print(sys.getrecursionlimit())          # thường là 1000


def chain_depth(k):
    return 0 if k == 0 else 1 + chain_depth(k - 1)


try:
    chain_depth(5000)
except RecursionError:
    print("RecursionError: đệ quy sâu 5000 vượt giới hạn mặc định")

1000
RecursionError: đệ quy sâu 5000 vượt giới hạn mặc định


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với ví dụ 1: `graph = [[1,2],[2,3],[5],[0],[5],[],[]]`. Các cạnh: `0→1, 0→2, 1→2, 1→3, 2→5, 3→0, 4→5`.

**Bước 1 – nút cuối.** `5` và `6` không có cạnh ra ⇒ **an toàn** (đường đi dừng ngay).

**Bước 2 – nút chỉ dẫn tới nút an toàn.**

- `2 → 5`: mọi nút kề của `2` (chỉ có `5`) an toàn ⇒ `2` **an toàn**.
- `4 → 5` ⇒ `4` **an toàn**.

**Bước 3 – các nút còn lại.** `0 → 1, 2`; `1 → 2, 3`; `3 → 0`. Nút `0` có nút kề `1` **chưa an toàn**; `1` có nút kề `3` chưa an toàn; `3` có nút kề `0` chưa an toàn. Ta bị **kẹt**: ba nút này **nhìn nhau** (`0 → 1 → 3 → 0`) nên không nút nào "chứng minh" được mình an toàn ⇒ chúng **không an toàn** (chính là chu trình).

Kết quả: `[2, 4, 5, 6]`.

**Quy tắc gọt (peeling):** *nút an toàn ⇔ mọi nút kề đều an toàn* (nút cuối thỏa vì không có nút kề nào). Ta lặp "tìm nút mà mọi nút kề đã an toàn" cho tới khi không còn nút nào như vậy. Những nút còn lại đều nằm trên hoặc dẫn tới chu trình.

In [8]:
# Gọt tay theo quy tắc: lặp tới khi không còn thay đổi
graph = ex1
safe = set()
changed = True
step = 0
while changed:
    changed = False
    for u in range(len(graph)):
        if u not in safe and all(v in safe for v in graph[u]):
            safe.add(u)
            changed = True
            step += 1
            print(f"bước {step}: nút {u} an toàn (nút kề {graph[u]} đều an toàn)")
print(sorted(safe))     # [2, 4, 5, 6]

bước 1: nút 5 an toàn (nút kề [] đều an toàn)
bước 2: nút 6 an toàn (nút kề [] đều an toàn)
bước 3: nút 2 an toàn (nút kề [5] đều an toàn)
bước 4: nút 4 an toàn (nút kề [5] đều an toàn)
[2, 4, 5, 6]


## 5. Cách 1 — Brute Force / baseline: bao đóng đạt tới

**Ý tưởng.** Dùng thẳng định nghĩa "không an toàn ⇔ đạt tới một chu trình":

1. Với **mỗi** nút `u`, tính tập `reach[u]` các nút đạt tới được từ `u` bằng **ít nhất một cạnh** (DFS/BFS).
2. `u` **nằm trên chu trình** nếu `u ∈ reach[u]` (quay về chính nó).
3. `u` **an toàn** nếu **không** nút nào trong `{u} ∪ reach[u]` nằm trên chu trình.
4. Trả về các `u` an toàn theo thứ tự tăng dần (duyệt `u` từ `0` đến `n - 1`).

In [9]:
def eventual_safe_bruteforce(graph):
    """Cách 1: bao đóng đạt tới. Time O(n * (n + m))."""
    n = len(graph)
    reach = [reachable(graph, u) for u in range(n)]
    on_cycle = [u in reach[u] for u in range(n)]
    return [u for u in range(n)
            if not on_cycle[u] and not any(on_cycle[v] for v in reach[u])]


for g, expected in examples + extra_examples:
    got = eventual_safe_bruteforce(g)
    print(got == expected, got)

True [2, 4, 5, 6]
True [4]
True [0]
True []
True []
True [1, 3]


**Độ phức tạp.** Mỗi nút chạy một lần DFS tốn `O(n + m)` ⇒ Time `O(n · (n + m))`. Với `n = 10^4`, `m = 4·10^4` là cỡ `5·10^8` bước: **quá chậm**. Space `O(n²)` nếu lưu mọi tập `reach` (ở đây là `n` tập, mỗi tập tới `n` phần tử).

**Nút thắt.** Ta **tính lại** cùng những đoạn đường đi nhiều lần: nếu `5` đã được biết an toàn thì mỗi nút trỏ tới `5` không cần đi lại. Đây là chỗ để **dùng kết quả của nút kề** (ghi nhớ) hoặc **gọt dần từ nút cuối** (lan ngược).

## 6. Ý tưởng thuật toán / bước đột phá

Hai cách khai thác "nút an toàn ⇔ mọi nút kề an toàn":

1. **Ghi nhớ khi DFS (ba màu).** DFS từ `u`: tô xám `u`; với mỗi nút kề `v`: nếu `v` xám (đang trên đường đi hoặc đã biết là xấu) thì `u` **không an toàn**; nếu `v` trắng thì đệ quy; nếu `v` không an toàn thì `u` cũng không an toàn. Nếu mọi nút kề an toàn thì tô đen `u`. Mỗi nút được xử lý **một lần** ⇒ `O(n + m)`.
2. **Gọt từ nút cuối bằng đồ thị đảo chiều (Kahn).** Nút an toàn đầu tiên là nút cuối (bậc ra `0`). Khi một nút `v` được xác nhận an toàn, **mọi nút `u` trỏ tới `v`** (tức `u ∈ rg[v]`) bớt đi một nút kề "chưa chắc". Khi bậc ra còn lại của `u` về `0`, **mọi nút kề của `u` đều an toàn** ⇒ `u` an toàn, đưa vào hàng đợi. Đây chính là **sắp xếp tô-pô** trên đồ thị đảo.

Hai cách trên không cần liệt kê chu trình. Một cách **nhìn khác** là tìm hẳn các chu trình bằng **thành phần liên thông mạnh (SCC)** (Tarjan/Kosaraju): nút nằm trong SCC có `≥ 2` nút hoặc có khuyên là **nằm trên chu trình**; sau đó lan ngược để đánh dấu các nút **dẫn tới** chúng; phần còn lại an toàn.

## 7. Các cách tiếp theo

### Cách 2 — DFS tô ba màu (có ghi nhớ)

**Giải thích từng đoạn.**

- `color[u]`: `0` trắng, `1` xám, `2` đen. **Quy ước quan trọng:** nút **không an toàn vẫn giữ màu xám mãi mãi** (ta không tô đen), nhờ vậy lần thăm sau chỉ cần nhìn màu là biết.
- `safe(u)`: nếu `color[u] != 0` trả `color[u] == 2` (đen ⇒ an toàn; xám ⇒ đang thăm hoặc không an toàn). Ngược lại tô xám `u`; với mỗi `v` trong `graph[u]`, nếu `not safe(v)` thì `return False` (để `u` xám); nếu hết vòng thì tô đen `u` và `return True`.
- Kết quả: các `u` có `color[u] == 2`, duyệt `u` từ nhỏ đến lớn nên đã **sắp tăng dần**.

In [10]:
def eventual_safe_dfs(graph):
    """Cách 2: DFS ba màu (đệ quy)."""
    n = len(graph)
    color = [0] * n                 # 0 trắng, 1 xám (đang thăm / không an toàn), 2 đen (an toàn)

    def safe(u):
        if color[u]:
            return color[u] == 2
        color[u] = 1
        for v in graph[u]:
            if not safe(v):
                return False        # u giữ màu xám = không an toàn
        color[u] = 2
        return True

    return [u for u in range(n) if safe(u)]


for g, expected in examples + extra_examples:
    assert eventual_safe_dfs(g) == expected
print("Cách 2 đúng trên các ví dụ")

Cách 2 đúng trên các ví dụ


In [11]:
def eventual_safe_dfs_iter(graph):
    """Cách 2b: cùng ý tưởng nhưng dùng ngăn xếp tường minh (không giới hạn đệ quy)."""
    n = len(graph)
    color = [0] * n
    for s in range(n):
        if color[s]:
            continue
        color[s] = 1
        stack = [(s, 0)]            # (nút, chỉ số nút kề kế tiếp cần xét)
        while stack:
            u, i = stack[-1]
            if i < len(graph[u]):
                stack[-1] = (u, i + 1)
                v = graph[u][i]
                if color[v] == 0:
                    color[v] = 1
                    stack.append((v, 0))
                elif color[v] == 1:
                    stack.clear()   # v xám: mọi nút trên ngăn xếp đạt tới v nên không an toàn (giữ xám)
                # color[v] == 2: v an toàn, bỏ qua
            else:
                color[u] = 2        # mọi nút kề an toàn
                stack.pop()
    return [u for u in range(n) if color[u] == 2]


for g, expected in examples + extra_examples:
    assert eventual_safe_dfs_iter(g) == expected
print("Cách 2b (không đệ quy) đúng trên các ví dụ")

Cách 2b (không đệ quy) đúng trên các ví dụ


**Độ phức tạp Cách 2:** mỗi nút đổi màu từ trắng sang xám **một lần** và mỗi cạnh được xét tối đa một lần ⇒ Time `O(n + m)`, Space `O(n)` (mảng màu) cộng độ sâu đệ quy tới `O(n)` (bản 2b dùng ngăn xếp tường minh).

---

### Cách 3 — Kahn trên đồ thị đảo chiều (sắp xếp tô-pô) — cách chuẩn

**Giải thích từng đoạn.**

- `rg`: đồ thị đảo chiều. `outdeg[i] = len(graph[i])`.
- `q`: hàng đợi các nút có `outdeg == 0` (nút cuối).
- Lấy `u` ra khỏi `q` (nút an toàn). Với mỗi `p ∈ rg[u]` (nút trỏ tới `u`): `outdeg[p] -= 1`; nếu `outdeg[p] == 0` thì mọi nút kề của `p` đã an toàn ⇒ thêm `p` vào `q`.
- Khi `q` rỗng, **các nút có `outdeg == 0`** chính là các nút an toàn (cả nút cuối ban đầu lẫn nút được thêm sau). Duyệt `i` từ `0` đến `n - 1` để lấy danh sách **tăng dần**. (Thứ tự thêm vào hàng đợi **không** tăng dần nên không trả luôn `q`.)

In [12]:
def eventual_safe_kahn_demo(graph):
    n = len(graph)
    rg = reverse_graph(graph)
    outdeg = [len(vs) for vs in graph]
    q = deque(i for i in range(n) if outdeg[i] == 0)
    print("outdeg ban đầu:", outdeg, "| hàng đợi:", list(q))
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for p in rg[u]:
            outdeg[p] -= 1
            print(f"  lấy {u}: nút {p} trỏ tới {u}, outdeg[{p}] -> {outdeg[p]}", "-> thêm vào hàng đợi" if outdeg[p] == 0 else "")
            if outdeg[p] == 0:
                q.append(p)
    print("thứ tự phát hiện:", order, "| outdeg cuối:", outdeg)
    return [i for i in range(n) if outdeg[i] == 0]


print(eventual_safe_kahn_demo(ex1))

outdeg ban đầu: [2, 2, 1, 1, 1, 0, 0] | hàng đợi: [5, 6]
  lấy 5: nút 2 trỏ tới 5, outdeg[2] -> 0 -> thêm vào hàng đợi
  lấy 5: nút 4 trỏ tới 5, outdeg[4] -> 0 -> thêm vào hàng đợi
  lấy 2: nút 0 trỏ tới 2, outdeg[0] -> 1 
  lấy 2: nút 1 trỏ tới 2, outdeg[1] -> 1 
thứ tự phát hiện: [5, 6, 2, 4] | outdeg cuối: [1, 1, 0, 1, 0, 0, 0]
[2, 4, 5, 6]


In [13]:
class Solution3:
    def eventualSafeNodes(self, graph: List[List[int]]) -> List[int]:
        n = len(graph)
        rg = [[] for _ in range(n)]
        outdeg = [0] * n
        for i, vs in enumerate(graph):
            outdeg[i] = len(vs)
            for j in vs:
                rg[j].append(i)
        q = deque(i for i in range(n) if outdeg[i] == 0)
        while q:
            u = q.popleft()
            for p in rg[u]:
                outdeg[p] -= 1
                if outdeg[p] == 0:
                    q.append(p)
        return [i for i in range(n) if outdeg[i] == 0]


for g, expected in examples + extra_examples:
    assert Solution3().eventualSafeNodes(g) == expected
print("Cách 3 đúng trên các ví dụ")

Cách 3 đúng trên các ví dụ


**Độ phức tạp Cách 3:** dựng `rg` tốn `O(n + m)`; mỗi nút vào hàng đợi tối đa một lần và mỗi cạnh `p → u` được xét đúng một lần khi `u` bị lấy ra ⇒ Time `O(n + m)`, Space `O(n + m)`. **Không đệ quy**, chạy tốt với chuỗi dài `10^4`.

---

### Cách 4 — Tarjan tìm SCC rồi lan truyền ngược

**Ý tưởng.** Dùng thuật toán **Tarjan** (cài **không đệ quy**) để chia đồ thị thành các **thành phần liên thông mạnh (SCC)**: hai nút cùng SCC nếu đi được tới nhau. Một nút **nằm trên chu trình** ⇔ SCC của nó có `≥ 2` nút, hoặc nó có **khuyên**. Gọi đó là các nút **chu trình**. Một nút **không an toàn** ⇔ nó **đạt tới** một nút chu trình ⇔ nó được tìm thấy khi **BFS trên đồ thị đảo** bắt đầu từ mọi nút chu trình. An toàn là các nút còn lại.

Tarjan: mỗi nút có `index` (thứ tự khám phá) và `low` (chỉ số nhỏ nhất đạt được qua một cạnh ngược); khi `low[v] == index[v]` thì `v` là "gốc" một SCC và ta lấy các nút trên ngăn xếp tới `v` ra làm một SCC.

In [14]:
def tarjan_scc(graph):
    """Tarjan không đệ quy: trả danh sách các SCC (mỗi SCC là list nút)."""
    n = len(graph)
    index = [-1] * n
    low = [0] * n
    on_stack = [False] * n
    st = []
    comps = []
    counter = 0
    for s in range(n):
        if index[s] != -1:
            continue
        index[s] = low[s] = counter
        counter += 1
        st.append(s)
        on_stack[s] = True
        work = [(s, 0)]
        while work:
            v, i = work[-1]
            if i < len(graph[v]):
                work[-1] = (v, i + 1)
                w = graph[v][i]
                if index[w] == -1:
                    index[w] = low[w] = counter
                    counter += 1
                    st.append(w)
                    on_stack[w] = True
                    work.append((w, 0))
                elif on_stack[w]:
                    low[v] = min(low[v], index[w])
            else:
                work.pop()
                if work:
                    parent = work[-1][0]
                    low[parent] = min(low[parent], low[v])
                if low[v] == index[v]:
                    comp = []
                    while True:
                        w = st.pop()
                        on_stack[w] = False
                        comp.append(w)
                        if w == v:
                            break
                    comps.append(comp)
    return comps


print(tarjan_scc(ex1))       # có một SCC {0, 1, 3} và các SCC một nút

[[5], [2], [3, 1, 0], [4], [6]]


In [15]:
def eventual_safe_scc(graph):
    """Cách 4: SCC + lan truyền ngược từ các nút chu trình."""
    n = len(graph)
    cyclic = []
    for comp in tarjan_scc(graph):
        if len(comp) > 1:
            cyclic.extend(comp)
        elif comp[0] in graph[comp[0]]:          # khuyên
            cyclic.append(comp[0])
    rg = reverse_graph(graph)
    unsafe = [False] * n
    q = deque()
    for c in cyclic:
        unsafe[c] = True
        q.append(c)
    while q:                                     # mọi nút đạt tới chu trình đều không an toàn
        u = q.popleft()
        for p in rg[u]:
            if not unsafe[p]:
                unsafe[p] = True
                q.append(p)
    return [u for u in range(n) if not unsafe[u]]


for g, expected in examples + extra_examples:
    assert eventual_safe_scc(g) == expected
print("Cách 4 đúng trên các ví dụ")

Cách 4 đúng trên các ví dụ


**Độ phức tạp Cách 4:** Tarjan `O(n + m)`, BFS ngược `O(n + m)` ⇒ Time `O(n + m)`, Space `O(n + m)`. Dài và nhiều chi tiết hơn Kahn, nhưng cho thêm **thông tin cấu trúc**: các SCC và các nút nằm trên chu trình (hữu ích cho biến thể ở mục 15).

Tổng kết: Cách 2 (DFS ghi nhớ) và Cách 3 (gọt từ nút cuối) dựa trên "nút an toàn ⇔ mọi nút kề an toàn"; Cách 4 dựa trên "không an toàn ⇔ đạt tới chu trình"; Cách 1 là định nghĩa thô.

## 8. TRACE từng bước

**Bảng trạng thái cho Cách 3 (Kahn) với ví dụ 1.** Cạnh đảo: `rg = [[3],[0],[0,1],[1],[],[2,4],[]]` (tức `rg[j]` là các nút trỏ tới `j`).

| Bước | Lấy ra `u` | Duyệt `p ∈ rg[u]` | `outdeg` sau | Hàng đợi sau |
|---|---|---|---|---|
| khởi tạo | | | `[2,2,1,1,1,0,0]` | `[5, 6]` |
| 1 | `5` | `p = 2`: `1 → 0` (thêm); `p = 4`: `1 → 0` (thêm) | `[2,2,0,1,0,0,0]` | `[6, 2, 4]` |
| 2 | `6` | `rg[6]` rỗng | không đổi | `[2, 4]` |
| 3 | `2` | `p = 0`: `2 → 1`; `p = 1`: `2 → 1` | `[1,1,0,1,0,0,0]` | `[4]` |
| 4 | `4` | `rg[4]` rỗng | không đổi | `[]` |

Hàng đợi rỗng. Các nút có `outdeg == 0`: `2, 4, 5, 6` ⇒ **`[2, 4, 5, 6]`**. Các nút `0, 1, 3` còn `outdeg = 1` vì mỗi nút vẫn trỏ tới một nút của chu trình.

Hàm `trace_dfs` dưới đây in diễn biến của **Cách 2 (DFS ba màu)** trên cùng ví dụ để thấy cách một nút bị "kẹt" ở màu xám.

In [16]:
def trace_dfs(graph):
    n = len(graph)
    color = [0] * n
    names = {0: "trắng", 1: "xám", 2: "đen"}

    def safe(u, depth=0):
        pad = "  " * depth
        if color[u]:
            print(f"{pad}{u}: đã {names[color[u]]} -> {'an toàn' if color[u] == 2 else 'không an toàn/đang thăm'}")
            return color[u] == 2
        color[u] = 1
        print(f"{pad}vào {u} (xám), nút kề {graph[u]}")
        for v in graph[u]:
            if not safe(v, depth + 1):
                print(f"{pad}{u}: nút kề {v} không an toàn -> {u} giữ màu xám (không an toàn)")
                return False
        color[u] = 2
        print(f"{pad}{u}: mọi nút kề an toàn -> đen")
        return True

    res = []
    for u in range(n):
        print(f"--- bắt đầu từ {u}")
        if safe(u):
            res.append(u)
    print("kết quả:", res, "| màu cuối:", color)
    return res


assert trace_dfs(ex1) == [2, 4, 5, 6]

--- bắt đầu từ 0
vào 0 (xám), nút kề [1, 2]
  vào 1 (xám), nút kề [2, 3]
    vào 2 (xám), nút kề [5]
      vào 5 (xám), nút kề []
      5: mọi nút kề an toàn -> đen
    2: mọi nút kề an toàn -> đen
    vào 3 (xám), nút kề [0]
      0: đã xám -> không an toàn/đang thăm
    3: nút kề 0 không an toàn -> 3 giữ màu xám (không an toàn)
  1: nút kề 3 không an toàn -> 1 giữ màu xám (không an toàn)
0: nút kề 1 không an toàn -> 0 giữ màu xám (không an toàn)
--- bắt đầu từ 1
1: đã xám -> không an toàn/đang thăm
--- bắt đầu từ 2
2: đã đen -> an toàn
--- bắt đầu từ 3
3: đã xám -> không an toàn/đang thăm
--- bắt đầu từ 4
vào 4 (xám), nút kề [5]
  5: đã đen -> an toàn
4: mọi nút kề an toàn -> đen
--- bắt đầu từ 5
5: đã đen -> an toàn
--- bắt đầu từ 6
vào 6 (xám), nút kề []
6: mọi nút kề an toàn -> đen
kết quả: [2, 4, 5, 6] | màu cuối: [1, 1, 2, 1, 2, 2, 2]


**Nhận xét từ trace DFS:** từ `0`: vào `0` (xám) → `1` (xám) → `2` (xám) → `5` (xám, không có nút kề ⇒ đen) ⇒ `2` đen. Quay lại `1`: nút kề `3` → vào `3` (xám) → nút kề `0` **đang xám** ⇒ chu trình ⇒ `3`, `1`, `0` đều giữ màu xám và **không an toàn**. Về sau khi `u = 1, 3` được xét lại, chỉ cần nhìn màu xám là biết ngay.

## 9. Code nộp LeetCode

Ta nộp **Cách 3 (Kahn trên đồ thị đảo chiều)**: `O(n + m)`, **không đệ quy** nên an toàn với đồ thị `10^4` nút dạng chuỗi dài. `deque` thường đã được import sẵn trên LeetCode; khi nộp hãy thêm `from collections import deque` ở đầu nếu cần.

In [17]:
class Solution:
    def eventualSafeNodes(self, graph: List[List[int]]) -> List[int]:
        n = len(graph)
        rg = [[] for _ in range(n)]          # đồ thị đảo chiều: rg[j] = các nút trỏ tới j
        outdeg = [0] * n
        for i, vs in enumerate(graph):
            outdeg[i] = len(vs)
            for j in vs:
                rg[j].append(i)
        q = deque(i for i in range(n) if outdeg[i] == 0)      # nút cuối
        while q:
            u = q.popleft()                                    # u an toàn
            for p in rg[u]:
                outdeg[p] -= 1                                 # p bớt một nút kề chưa chắc chắn
                if outdeg[p] == 0:                             # mọi nút kề của p an toàn
                    q.append(p)
        return [i for i in range(n) if outdeg[i] == 0]         # tăng dần theo đề


for g, expected in examples + extra_examples:
    assert Solution().eventualSafeNodes(g) == expected
print("Code nộp đúng trên 6 ví dụ")

Code nộp đúng trên 6 ví dụ


## 10. Vì sao thuật toán đúng

Gọi **tập an toàn** `S` là tập các nút không đạt tới chu trình nào (định nghĩa hiệu dụng của đề, vì "mọi đường đi dẫn tới nút cuối" ⇔ "không có đường đi vô hạn" ⇔ "không đạt tới chu trình" trong đồ thị hữu hạn).

**Khẳng định:** thuật toán Kahn trên đồ thị đảo kết thúc với `outdeg[u] == 0` **đúng** cho các nút `u ∈ S`.

**(a) Mọi nút được thêm vào hàng đợi đều an toàn (sound).** Quy nạp theo thứ tự thêm vào. Nút đầu là nút cuối (không đi ra đâu, an toàn). Giả sử các nút thêm trước `p` đều an toàn. `p` chỉ được thêm khi `outdeg[p]` về `0`, tức **mọi nút kề** `v` của `p` đã bị lấy ra (mỗi cạnh `p → v` giảm `outdeg[p]` đúng một lần, khi `v` được lấy ra) và do đó `v` đã vào hàng đợi trước, nên an toàn. Mọi đường đi từ `p` đi qua một nút kề an toàn, nên không đạt tới chu trình ⇒ `p` an toàn.

**(b) Mọi nút an toàn đều được thêm vào (complete).** Đồ thị con những nút đạt tới được từ một nút an toàn là **không có chu trình** (DAG), nên mọi nút an toàn có **độ dài đường đi dài nhất** tới nút cuối hữu hạn. Quy nạp theo số đó: độ dài `0` là nút cuối: có trong hàng đợi ban đầu. Nút `u` có độ dài `k > 0`: mọi nút kề của nó có độ dài `< k` (an toàn) nên đã được thêm vào hàng đợi (giả thiết quy nạp), nên sớm hay muộn đều bị lấy ra và giảm `outdeg[u]` đúng `len(graph[u])` lần ⇒ `outdeg[u] = 0` ⇒ `u` được thêm.

**(c) Nút không an toàn không bao giờ về 0.** Hệ quả của (a): nếu nó về `0` thì an toàn. Nút trên chu trình luôn còn ít nhất một nút kề (nút kế tiếp trên chu trình) **không bao giờ được lấy ra** (nếu nút đầu tiên của chu trình được lấy ra thì các nút kề của nó đã được lấy ra từ trước, mâu thuẫn).

**(d) Đáp án tăng dần.** Ta liệt kê `i` từ `0` đến `n - 1` rồi lọc `outdeg[i] == 0`, nên thứ tự là tăng dần (thứ tự thêm vào hàng đợi thì không, nên **không** trả trực tiếp hàng đợi).

**Các cách khác.** Cách 2 áp dụng cùng nguyên lý "an toàn ⇔ mọi nút kề an toàn" nhưng từ trên xuống bằng đệ quy và màu: nút xám gặp lại nghĩa là có đường đi vòng ⇒ chu trình. Một nút giữ màu xám mãi nghĩa là đã biết không an toàn. Cách 4 dùng đặc trưng: nút trên chu trình ⇔ SCC lớn hoặc có khuyên; nút không an toàn ⇔ đạt tới nút như vậy. Cách 1 trực tiếp dùng định nghĩa nên là oracle. Ở mục 12 ta đối chiếu mọi cách trên hàng nghìn đồ thị ngẫu nhiên và kiểm tra các **tính chất bắt buộc** của `S`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `m` là số cạnh.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Bao đóng đạt tới | `O(n · (n + m))` | `O(n²)` | oracle, quá chậm khi `n = 10^4` |
| 2. DFS ba màu | `O(n + m)` | `O(n)` + đệ quy | bản 2b dùng ngăn xếp tường minh |
| 3. Kahn trên đồ thị đảo (nộp) | `O(n + m)` | `O(n + m)` | không đệ quy |
| 4. Tarjan + lan ngược | `O(n + m)` | `O(n + m)` | cho thêm thông tin SCC |

Cận dưới `Ω(n + m)` vì phải đọc toàn bộ đồ thị. Với `n ≤ 10^4`, `m ≤ 4·10^4`, mọi cách tuyến tính chạy tức thì.

## 12. Bộ test

### 12.1. Ví dụ đề, ca tính tay và biên

| `graph` | Mô tả | Kết quả (tính tay) |
|---|---|---|
| `[[]]` | một nút cuối | `[0]` |
| `[[0]]` | một khuyên | `[]` |
| `[[1],[1]]` | `0` dẫn vào khuyên ở `1` | `[]` |
| `[[1],[2],[]]` | chuỗi `0 → 1 → 2` | `[0,1,2]` |
| `[[1],[0]]` | hai nút trỏ nhau | `[]` |
| `[[],[0]]` | `1 → 0`, `0` là nút cuối | `[0,1]` |
| `[[1,2],[3],[3,4],[],[2]]` | `2 ↔ 4` là chu trình, `0` dẫn vào nó | `[1,3]` |
| `[[1,2],[3],[3],[]]` | hình thoi (DAG, nút `3` được thăm hai lần) | `[0,1,2,3]` |
| `[[1],[2],[0],[]]` | chu trình ba nút và một nút cô lập | `[3]` |
| `[[1,2],[2],[]]` | DAG có cạnh "tắt" | `[0,1,2]` |

In [18]:
hand_cases = [
    ([[]], [0]),
    ([[0]], []),
    ([[1], [1]], []),
    ([[1], [2], []], [0, 1, 2]),
    ([[1], [0]], []),
    ([[], [0]], [0, 1]),
    ([[1, 2], [3], [3, 4], [], [2]], [1, 3]),
    ([[1, 2], [3], [3], []], [0, 1, 2, 3]),
    ([[1], [2], [0], []], [3]),
    ([[1, 2], [2], []], [0, 1, 2]),
]
all_cases = examples + extra_examples + hand_cases

solvers = {
    "Cách 1 bao đóng": eventual_safe_bruteforce,
    "Cách 2 DFS ba màu": eventual_safe_dfs,
    "Cách 2b DFS ngăn xếp": eventual_safe_dfs_iter,
    "Cách 3 Kahn": Solution3().eventualSafeNodes,
    "Cách 4 Tarjan SCC": eventual_safe_scc,
    "Code nộp": Solution().eventualSafeNodes,
}
for g, expected in all_cases:
    for name, f in solvers.items():
        assert f(g) == expected, (name, g, f(g), expected)
print("Tất cả", len(all_cases), "ca đều đúng với mọi cách")

Tất cả 16 ca đều đúng với mọi cách


### 12.2. Các tính chất bắt buộc của tập an toàn `S`

Một tập `S` là đáp án đúng **khi và chỉ khi** thỏa cả ba tính chất sau (kiểm tra **độc lập** với mọi thuật toán):

1. **Đóng theo cạnh ra:** nếu `u ∈ S` thì mọi nút kề của `u` đều thuộc `S` (an toàn ⇒ mọi nút kề an toàn).
2. **Không có chu trình bên trong `S`:** đồ thị con trên `S` là DAG (kiểm tra bằng sắp xếp tô-pô).
3. **Phần bù đầy đủ "xấu":** mỗi nút `u ∉ S` có **ít nhất một** nút kề cũng `∉ S`. Khi đó từ `u` ta đi mãi trong phần bù, mà phần bù hữu hạn nên phải lặp nút ⇒ đạt tới một chu trình ⇒ `u` thật sự không an toàn.

Ba tính chất này xác định **duy nhất** `S`.

In [19]:
def check_safe_set(graph, result):
    n = len(graph)
    S = set(result)
    assert result == sorted(result) and len(S) == len(result)             # tăng ngặt
    assert all(0 <= u < n for u in S)
    for u in S:                                                            # 1
        assert all(v in S for v in graph[u]), ("không đóng", u)
    indeg = {u: 0 for u in S}                                              # 2: Kahn trên đồ thị con
    for u in S:
        for v in graph[u]:
            indeg[v] += 1
    q = [u for u in S if indeg[u] == 0]
    seen = 0
    while q:
        u = q.pop()
        seen += 1
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    assert seen == len(S), "có chu trình bên trong S"
    for u in range(n):                                                     # 3
        if u not in S:
            assert any(v not in S for v in graph[u]), ("nút xấu không có nút kề xấu", u)


for g, expected in all_cases:
    check_safe_set(g, expected)
print("Các tính chất đúng trên mọi đáp án mong đợi đã biết")

Các tính chất đúng trên mọi đáp án mong đợi đã biết


### 12.3. Đối chiếu ngẫu nhiên có seed, đồ thị nhỏ

Đồ thị ngẫu nhiên `1..9` nút, mật độ cạnh khác nhau, **có khuyên** (đúng ràng buộc: `graph[i]` tăng ngặt, không cạnh lặp). So cả sáu cách và kiểm tra các tính chất. Ta cũng **vét cạn toàn bộ** đồ thị `n = 3` (`2^9 = 512` đồ thị) và `n = 4` (`2^16 = 65536` đồ thị).

In [20]:
def random_graph(rng, n, p):
    return [[j for j in range(n) if rng.random() < p] for _ in range(n)]


rng = random.Random(802)
for _ in range(3000):
    n = rng.randint(1, 9)
    g = random_graph(rng, n, rng.choice([0.1, 0.2, 0.35, 0.5, 0.8]))
    ref = eventual_safe_bruteforce(g)
    for name, f in solvers.items():
        assert f(g) == ref, (name, g)
    check_safe_set(g, ref)
print("3000 đồ thị ngẫu nhiên (seed 802): 6 cách khớp và thỏa các tính chất")

total = 0
for n in (1, 2, 3, 4):
    pairs = [(i, j) for i in range(n) for j in range(n)]
    for mask in range(1 << len(pairs)):
        g = [[] for _ in range(n)]
        for k, (i, j) in enumerate(pairs):
            if mask >> k & 1:
                g[i].append(j)
        ref = eventual_safe_bruteforce(g)
        assert Solution().eventualSafeNodes(g) == ref == eventual_safe_scc(g) == eventual_safe_dfs(g)
        total += 1
print("Vét cạn mọi đồ thị n = 1..4:", total, "đồ thị, tất cả khớp")

3000 đồ thị ngẫu nhiên (seed 802): 6 cách khớp và thỏa các tính chất


Vét cạn mọi đồ thị n = 1..4: 66066 đồ thị, tất cả khớp


### 12.4. Cỡ lớn (đúng ràng buộc) và đồ thị cực đoan

`n = 10^4`, `m` tới `4·10^4`. Cách 1 chạy chậm nên bỏ qua. Cách 2 (đệ quy) cần nới giới hạn đệ quy cho chuỗi dài; code nộp thì không. Các dạng cực đoan: chuỗi `0 → 1 → ... → 9999` (tất cả an toàn), chuỗi có khuyên ở cuối (tất cả không an toàn), một chu trình lớn, hình sao, đồ thị ngẫu nhiên thưa/dày.

In [21]:
sys.setrecursionlimit(60000)              # chỉ để thử Cách 2 đệ quy trên chuỗi dài

N = 10 ** 4
chain_safe = [[i + 1] for i in range(N - 1)] + [[]]                  # 0 -> 1 -> ... -> N-1 (nút cuối)
chain_loop = [[i + 1] for i in range(N - 1)] + [[N - 1]]             # nút cuối có khuyên
big_cycle = [[(i + 1) % N] for i in range(N)]
star_out = [list(range(1, N))] + [[] for _ in range(N - 1)]          # 0 -> mọi nút khác (nhưng m <= 4e4 nên chỉ để thử)
star_in = [[0] for _ in range(N)]
star_in[0] = []                                                       # mọi nút trỏ vào 0 (nút cuối)

rng = random.Random(8020)
big_random = [sorted(set(rng.randrange(N) for _ in range(rng.choice([0, 1, 2, 4])))) for _ in range(N)]
assert sum(len(v) for v in big_random) <= 4 * 10 ** 4

cases = {
    "chuỗi an toàn": (chain_safe, list(range(N))),
    "chuỗi có khuyên ở cuối": (chain_loop, []),
    "một chu trình lớn": (big_cycle, []),
    "sao đi ra": (star_out, list(range(N))),
    "sao đi vào": (star_in, list(range(N))),
    "ngẫu nhiên lớn": (big_random, None),
}
for name, (g, expected) in cases.items():
    ref = Solution().eventualSafeNodes(g)
    if expected is not None:
        assert ref == expected, name
    assert eventual_safe_dfs(g) == ref == eventual_safe_dfs_iter(g) == eventual_safe_scc(g) == Solution3().eventualSafeNodes(g)
    check_safe_set(g, ref)
    print(f"{name:<26} n = {len(g)}, m = {sum(map(len, g))}, số nút an toàn = {len(ref)}")

chuỗi an toàn              n = 10000, m = 9999, số nút an toàn = 10000
chuỗi có khuyên ở cuối     n = 10000, m = 10000, số nút an toàn = 0
một chu trình lớn          n = 10000, m = 10000, số nút an toàn = 0
sao đi ra                  n = 10000, m = 9999, số nút an toàn = 10000
sao đi vào                 n = 10000, m = 9999, số nút an toàn = 10000


ngẫu nhiên lớn             n = 10000, m = 17539, số nút an toàn = 3925


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ loại các nút **nằm trên** chu trình (quên các nút **dẫn tới** chu trình)

Nút `0` ở `[[1],[1]]` **không** nằm trên chu trình nào (chỉ `1` có khuyên) nhưng **mọi đường đi** từ `0` đều vào khuyên rồi quay vòng mãi nên `0` **không an toàn**.

In [22]:
def sai_chi_loai_nut_chu_trinh(graph):
    n = len(graph)
    reach = [reachable(graph, u) for u in range(n)]
    return [u for u in range(n) if u not in reach[u]]        # SAI: chỉ loại nút nằm trên chu trình


g = [[1], [1]]
print("sai :", sai_chi_loai_nut_chu_trinh(g))          # [0]
print("đúng:", Solution().eventualSafeNodes(g))        # []
assert sai_chi_loai_nut_chu_trinh(g) != Solution().eventualSafeNodes(g)

sai : [0]
đúng: []


**Vì sao sai:** một nút là an toàn khi **mọi** đường đi từ nó an toàn. Chỉ cần **một** đường đi dẫn vào chu trình là đủ để loại. **Sửa:** loại cả những nút **đạt tới** nút chu trình (DFS ghi nhớ hoặc Kahn trên đồ thị đảo).

### Lỗi 2: Dùng một tập `visited` (hai trạng thái) thay vì ba màu

Nếu coi "đã thăm" luôn là "có chu trình", một nút được thăm **hai lần** theo hai đường khác nhau (như đỉnh `3` trong hình thoi) sẽ bị nhầm là chu trình.

In [23]:
def sai_visited_hai_trang_thai(graph):
    n = len(graph)
    visited = [False] * n

    def safe(u):
        if visited[u]:
            return False                   # SAI: đã thăm = chu trình
        visited[u] = True
        return all(safe(v) for v in graph[u])

    return [u for u in range(n) if safe(u)]


diamond = [[1, 2], [3], [3], []]
print("sai :", sai_visited_hai_trang_thai(diamond))         # thiếu nút
print("đúng:", Solution().eventualSafeNodes(diamond))       # [0, 1, 2, 3]
assert sai_visited_hai_trang_thai(diamond) != [0, 1, 2, 3]

sai : []
đúng: [0, 1, 2, 3]


**Vì sao sai:** nút `3` đã **xong và an toàn** khi ta quay lại từ nhánh `2 → 3`; chỉ nút **đang nằm trên đường đi hiện tại** (xám) mới chứng tỏ chu trình. **Sửa:** ba màu: trắng / xám (đang thăm hoặc xấu) / đen (xong, an toàn).

### Lỗi 3: Chạy Kahn trên đồ thị **chưa đảo** (bậc vào)

Kahn kinh điển (sắp xếp tô-pô) trên đồ thị gốc gọt các nút có **bậc vào** `0` (nguồn). Nút an toàn lại là các nút **không dẫn tới chu trình** (ở phía **đích**), nên cần gọt theo **bậc ra** trên đồ thị đảo.

In [24]:
def sai_kahn_khong_dao(graph):
    n = len(graph)
    indeg = [0] * n
    for vs in graph:
        for v in vs:
            indeg[v] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    removed = []
    while q:
        u = q.popleft()
        removed.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return sorted(removed)                  # SAI: đây là các nút "không bị chu trình chặn đằng trước"


print("sai :", sai_kahn_khong_dao(ex1))              # [4, 6]  (chỉ gọt các nút nguồn)
print("đúng:", Solution().eventualSafeNodes(ex1))    # [2, 4, 5, 6]
assert sai_kahn_khong_dao(ex1) != [2, 4, 5, 6]

sai : [4, 6]
đúng: [2, 4, 5, 6]


**Vì sao sai:** gọt theo bậc vào tìm các nút **không bị chu trình đi tới** (ở phía nguồn), còn đề hỏi các nút **không đi tới được chu trình** (phía đích). **Sửa:** đảo chiều cạnh (hoặc dùng bậc ra và lan ngược qua `rg`).

### Lỗi 4: Quên sắp xếp kết quả

Nếu trả luôn thứ tự phát hiện của Kahn hoặc thứ tự hoàn thành của DFS, kết quả **không tăng dần**, trái với yêu cầu của đề.

In [25]:
def sai_khong_sap_xep(graph):
    n = len(graph)
    rg = reverse_graph(graph)
    outdeg = [len(vs) for vs in graph]
    q = deque(i for i in range(n) if outdeg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for p in rg[u]:
            outdeg[p] -= 1
            if outdeg[p] == 0:
                q.append(p)
    return order                            # SAI: thứ tự phát hiện, không tăng dần


print("sai :", sai_khong_sap_xep(ex1))              # [5, 6, 2, 4]
print("đúng:", Solution().eventualSafeNodes(ex1))   # [2, 4, 5, 6]
assert sai_khong_sap_xep(ex1) != [2, 4, 5, 6] and sorted(sai_khong_sap_xep(ex1)) == [2, 4, 5, 6]

sai : [5, 6, 2, 4]
đúng: [2, 4, 5, 6]


**Sửa:** `sorted(order)` hoặc liệt kê `i` từ `0` đến `n - 1` và lọc `outdeg[i] == 0` như code nộp.

### Lỗi 5: Bỏ qua **khuyên** `i → i`

Khuyên là chu trình độ dài `1`. Nếu bỏ qua cạnh `u → u` (cho rằng "tự trỏ vào mình không sao") thì nút có khuyên bị coi an toàn.

In [26]:
def sai_bo_qua_khuyen(graph):
    n = len(graph)
    color = [0] * n

    def safe(u):
        if color[u]:
            return color[u] == 2
        color[u] = 1
        for v in graph[u]:
            if v == u:
                continue                   # SAI: bỏ qua khuyên
            if not safe(v):
                return False
        color[u] = 2
        return True

    return [u for u in range(n) if safe(u)]


g = [[0], [0], []]                         # nút 0 có khuyên, 1 -> 0, 2 là nút cuối
print("sai :", sai_bo_qua_khuyen(g))                # [0, 1, 2]
print("đúng:", Solution().eventualSafeNodes(g))     # [2]
assert sai_bo_qua_khuyen(g) != Solution().eventualSafeNodes(g)

sai : [0, 1, 2]
đúng: [2]


**Vì sao sai:** từ `0` ta có thể đi `0 → 0 → 0 → ...` mãi, không bao giờ tới nút cuối. **Sửa:** đừng loại trừ `v == u`; DFS sẽ thấy `u` đang xám và nhận ra chu trình; Kahn thì `outdeg[0]` không bao giờ về `0` vì cạnh `0 → 0` chỉ được trừ khi `0` được lấy ra.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "mọi đường đi đều dẫn tới ...", "đạt tới chu trình", đồ thị có hướng ⇒ **phát hiện chu trình** (DFS ba màu) hoặc **sắp xếp tô-pô** (Kahn).
- **An toàn ⇔ không đạt tới chu trình:** đổi bài toán "mọi đường đi" thành bài toán "tồn tại một đường đi xấu" rồi lấy phần bù.
- **Đảo chiều cạnh + bậc ra** là mẹo đổi "nút phụ thuộc vào nút nào" thành "lan ngược": dùng khi đích là nút cuối.
- **Ba màu** cho DFS phát hiện chu trình trong đồ thị có hướng (hai màu chỉ đủ cho vô hướng).
- **Đồ thị lớn (`10^4`) trong Python:** tránh đệ quy sâu; chọn Kahn hoặc DFS ngăn xếp.
- **Nhớ sắp xếp** nếu đề yêu cầu thứ tự tăng và thuật toán không tự sinh thứ tự đó.
- **Kiểm chứng bằng vét cạn toàn bộ đồ thị nhỏ** (mọi tập cạnh với `n ≤ 4`) là cách rất mạnh để bắt lỗi biên (khuyên, nút cô lập).
- Bài liên quan: **Course Schedule I/II** (207, 210), **Course Schedule IV**, **Find Eventual Safe States** là trường hợp "gọt từ nút cuối" của sắp xếp tô-pô; **Number of Provinces** (SCC vô hướng).

## 15. Biến thể

1. **Trả về các nút không an toàn** hoặc **đếm** chúng: phần bù của đáp án.
2. **Tìm mọi nút nằm trên chu trình:** dùng SCC (Tarjan) như Cách 4.
3. **Khoảng cách tới nút cuối của từng nút an toàn:** "mức" trong Kahn theo lớp (độ dài đường đi dài nhất tới nút cuối).
4. **Course Schedule:** có thể hoàn thành mọi môn không (đồ thị không có chu trình)?
5. **Đồ thị trọng số / nhiều nút cuối đặc biệt:** đổi điều kiện "nút an toàn" thành "đi tới được tập đích".
6. **Nén đồ thị thành DAG các SCC** (condensation) và xử lý trên DAG.

In [27]:
def cycle_nodes(graph):
    """Biến thể 2: danh sách (tăng dần) các nút nằm trên một chu trình."""
    res = []
    for comp in tarjan_scc(graph):
        if len(comp) > 1 or comp[0] in graph[comp[0]]:
            res.extend(comp)
    return sorted(res)


def safe_levels(graph):
    """Biến thể 3: với mỗi nút an toàn, độ dài đường đi dài nhất tới nút cuối (nút cuối là 0)."""
    n = len(graph)
    rg = reverse_graph(graph)
    outdeg = [len(vs) for vs in graph]
    level = {}
    cur = [i for i in range(n) if outdeg[i] == 0]
    d = 0
    while cur:
        nxt = []
        for u in cur:
            level[u] = d
            for p in rg[u]:
                outdeg[p] -= 1
                if outdeg[p] == 0:
                    nxt.append(p)
        cur = nxt
        d += 1
    return level


print(cycle_nodes(ex1))        # [0, 1, 3]
print(safe_levels(ex1))        # {5: 0, 6: 0, 2: 1, 4: 1}
assert cycle_nodes(ex1) == [0, 1, 3]
assert safe_levels(ex1) == {5: 0, 6: 0, 2: 1, 4: 1}

[0, 1, 3]
{5: 0, 6: 0, 2: 1, 4: 1}


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bao đóng đạt tới | Nút xấu ⇔ đạt tới nút nằm trên chu trình; tính reach mỗi nút | `O(n · (n + m))` | `O(n²)` | Làm oracle, hiểu định nghĩa |
| 2. DFS ba màu | Ghi nhớ trạng thái; xám gặp lại ⇒ chu trình; nút xấu giữ xám | `O(n + m)` | `O(n)` + đệ quy | Đồ thị nhỏ hoặc dùng bản ngăn xếp |
| 3. Kahn trên đồ thị đảo | Gọt từ nút cuối: bậc ra về `0` ⇒ an toàn | `O(n + m)` | `O(n + m)` | **Nộp bài**: không đệ quy, dễ chứng minh |
| 4. Tarjan SCC + lan ngược | Tìm nút trên chu trình, lan ngược tới mọi nút dẫn tới chúng | `O(n + m)` | `O(n + m)` | Cần cấu trúc SCC, hoặc làm bài biến thể về chu trình |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().eventualSafeNodes([[1],[2],[],[0]])`; `Solution().eventualSafeNodes([[1],[0]])`; `Solution().eventualSafeNodes([[0]])`; `Solution().eventualSafeNodes([[],[0]])`; và `reverse_graph([[1],[2],[]])`.

**Bài 2 (tính tay).** Với `graph = [[1,2],[3],[3,4],[],[2]]`, vẽ các cạnh, tìm chu trình, liệt kê nút an toàn từng bước theo quy tắc "gọt" ở mục 4 và cho đáp án.

**Bài 3 (điền TODO).** Hoàn thành `safe_nodes_todo(graph)` theo Cách 3 (Kahn trên đồ thị đảo).

In [28]:
def safe_nodes_todo(graph):
    # TODO 1: n = len(graph); rg = [[] for _ in range(n)]; outdeg = [len(vs) for vs in graph]
    #         với mỗi i, j trong graph[i]: rg[j].append(i)
    # TODO 2: q = deque các i có outdeg[i] == 0
    # TODO 3: while q: u = q.popleft(); với p trong rg[u]: outdeg[p] -= 1; nếu về 0 thì q.append(p)
    # TODO 4: return [i for i in range(n) if outdeg[i] == 0]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn code mẫu, viết `safe_nodes_dfs(graph)` bằng DFS ba màu **không đệ quy** (ngăn xếp tường minh), trả về danh sách tăng dần.

**Bài 5 (biến thể / mini challenge).** (a) Viết `unsafe_count(graph)` trả về **số nút không an toàn** và kiểm tra `unsafe_count + số nút an toàn == n`. (b) Viết `has_cycle(graph)` trả về `True` nếu đồ thị có chu trình (giả sử đã có các hàm ở trên), kiểm chứng: đồ thị không có chu trình ⇔ mọi nút đều an toàn.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[0,1,2,3]` (chuỗi `3 → 0 → 1 → 2`, `2` là nút cuối); `[]`; `[]`; `[0,1]`; `[[], [0], [1]]`. Bài 3: bám sát mục 9.
- **Mức 2 (vừa):** Bài 2: các cạnh `0→1, 0→2, 1→3, 2→3, 2→4, 4→2`; chu trình `2 ↔ 4`. Nút cuối `3` an toàn; `1 → 3` nên `1` an toàn; `2` có nút kề `4` chưa an toàn, `4` có nút kề `2` chưa an toàn: kẹt; `0` có nút kề `2` chưa an toàn. Đáp án `[1, 3]`. Bài 4: nhớ **không tô đen** nút không an toàn: để nó xám.
- **Mức 3 (sâu):** Bài 5(a): `n - len(Solution().eventualSafeNodes(graph))`. Bài 5(b): đồ thị có chu trình ⇔ **tồn tại nút không an toàn** (nếu có chu trình, nút trên chu trình không an toàn; ngược lại nút không an toàn đạt tới chu trình), nên `has_cycle = len(safe) < n`; kiểm chứng bằng cách so với `cycle_nodes(graph) != []`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert Solution().eventualSafeNodes([[1], [2], [], [0]]) == [0, 1, 2, 3]
assert Solution().eventualSafeNodes([[1], [0]]) == []
assert Solution().eventualSafeNodes([[0]]) == []
assert Solution().eventualSafeNodes([[], [0]]) == [0, 1]
assert reverse_graph([[1], [2], []]) == [[], [0], [1]]

# Bài 2
g2 = [[1, 2], [3], [3, 4], [], [2]]
assert Solution().eventualSafeNodes(g2) == [1, 3]
assert set(max(tarjan_scc(g2), key=len)) == {2, 4}


# Bài 3
def safe_nodes_todo_ans(graph):
    n = len(graph)
    rg = [[] for _ in range(n)]
    outdeg = [len(vs) for vs in graph]
    for i, vs in enumerate(graph):
        for j in vs:
            rg[j].append(i)
    q = deque(i for i in range(n) if outdeg[i] == 0)
    while q:
        u = q.popleft()
        for p in rg[u]:
            outdeg[p] -= 1
            if outdeg[p] == 0:
                q.append(p)
    return [i for i in range(n) if outdeg[i] == 0]


# Bài 4
def safe_nodes_dfs(graph):
    n = len(graph)
    color = [0] * n
    for s in range(n):
        if color[s]:
            continue
        color[s] = 1
        stack = [(s, 0)]
        while stack:
            u, i = stack[-1]
            if i < len(graph[u]):
                stack[-1] = (u, i + 1)
                v = graph[u][i]
                if color[v] == 0:
                    color[v] = 1
                    stack.append((v, 0))
                elif color[v] == 1:
                    stack.clear()
            else:
                color[u] = 2
                stack.pop()
    return [u for u in range(n) if color[u] == 2]


# Bài 5
def unsafe_count(graph):
    return len(graph) - len(Solution().eventualSafeNodes(graph))


def has_cycle(graph):
    return len(Solution().eventualSafeNodes(graph)) < len(graph)


rng5 = random.Random(8021)
for _ in range(500):
    n = rng5.randint(1, 8)
    g = random_graph(rng5, n, rng5.choice([0.1, 0.25, 0.5]))
    ref = eventual_safe_bruteforce(g)
    assert safe_nodes_todo_ans(g) == safe_nodes_dfs(g) == ref
    assert unsafe_count(g) + len(ref) == n
    assert has_cycle(g) == (cycle_nodes(g) != [])
assert unsafe_count(ex1) == 3 and has_cycle(ex1)
assert not has_cycle([[1], [2], []])
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Kahn trên đồ thị đảo, gọt từ nút cuối)

```python
class Solution:
    def eventualSafeNodes(self, graph: List[List[int]]) -> List[int]:
        n = len(graph)
        rg = [[] for _ in range(n)]            # rg[j] = các nút trỏ tới j
        outdeg = [0] * n
        for i, vs in enumerate(graph):
            outdeg[i] = len(vs)
            for j in vs:
                rg[j].append(i)
        q = deque(i for i in range(n) if outdeg[i] == 0)     # nút cuối
        while q:
            u = q.popleft()
            for p in rg[u]:
                outdeg[p] -= 1
                if outdeg[p] == 0:
                    q.append(p)
        return [i for i in range(n) if outdeg[i] == 0]       # tăng dần
```

**Checklist**

- [ ] Nút **an toàn ⇔ không đạt tới chu trình** (không chỉ "không nằm trên chu trình")?
- [ ] Dùng **bậc ra** và **đồ thị đảo chiều** (không phải bậc vào trên đồ thị gốc)?
- [ ] Nếu dùng DFS: **ba màu**, nút không an toàn **giữ màu xám**, nút an toàn mới được tô đen?
- [ ] Khuyên `i → i` được tính là chu trình (không bị bỏ qua)?
- [ ] Kết quả **sắp tăng dần** (duyệt `i` từ `0` đến `n - 1`, không trả thứ tự hàng đợi)?
- [ ] Không đệ quy sâu `10^4` (Kahn hoặc DFS ngăn xếp), hoặc đã cân nhắc `setrecursionlimit`?
- [ ] Đã thử ví dụ 1 (`[2,4,5,6]`), ví dụ 2 (`[4]`), `[[0]]`, `[[1],[1]]`, một nút cuối `[[]]`?
- [ ] Skeleton đúng `class Solution` / `def eventualSafeNodes(self, graph: List[List[int]]) -> List[int]`?